In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

# Загрузка данных
df = pd.read_csv('/content/drive/MyDrive/train_2.csv')

rto_09 = df.loc[(df['new_id'] == 9754) & (df['Год'] == 2023) & (df['Месяц'] == 9), 'РТО'].values[0]
rto_11 = df.loc[(df['new_id'] == 9754) & (df['Год'] == 2023) & (df['Месяц'] == 11), 'РТО'].values[0]

mean_rto = (rto_09 + rto_11) / 2

df.loc[(df['new_id'] == 9754) & (df['Год'] == 2023) & (df['Месяц'] == 10), 'РТО'] = mean_rto

# Создаём дату
df['date'] = pd.to_datetime(df['Год'].astype(str) + '-' + df['Месяц'].astype(str) + '-01')

# Переименовываем колонки для удобства
df = df.rename(columns={
    'Среднее количество промо товаров в чеке': 'avg_promo',
    'Среднее количество товаров в чеке': 'avg_items',
    'Среднее количество отмен': 'avg_cancels',
    'Рабочие часы в день': 'work_hours',
    'Дата открытия, категориальный': 'store_age_cat',
    'Торговая площадь, категориальный': 'area_cat',
    'Населенный пункт': 'city',
    'Регион': 'region',
    'Численность населения': 'population',
    'Количество домохозяйств': 'households',
    'Трафик пеший, в час': 'foot_traffic',
    'Трафик авто, в час': 'car_traffic',
    'Маркетплейсы, доставки, постаматы (100 м)': 'marketplaces',
    'Медицинские уч. и аптеки (300 м)': 'medical',
    'Школы (300 м)': 'schools',
    'Остановки (300 м)': 'stops',
    'Продуктовые магазины (500 м)': 'grocery_competitors',
    'Пятерочки (500 м)': 'pyaterochka_nearby',
    'Количество касс': 'n_kass',
    'Флаг алкогольной лицензии': 'alcohol_flag',
    'РТО': 'rto'
})

# Сортировка — критически важно для лагов
df = df.sort_values(['new_id', 'date']).reset_index(drop=True)

# Проверка
print(f"Shape: {df.shape}")
print(f"Даты: {df['date'].min()} — {df['date'].max()}")
print(f"Магазинов: {df['new_id'].nunique()}")
print(f"\nПервые строки:")
df.head()

In [ ]:
eps = 1e-6

# 1) Календарные признаки
df['year'] = df['Год'].astype(np.int16)
df['month'] = df['Месяц'].astype(np.int8)

df['time_idx'] = (df['year'] - df['year'].min()) * 12 + (df['month'] - 1)
df['quarter'] = ((df['month'] - 1) // 3 + 1).astype(np.int8)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['is_december'] = (df['month'] == 12).astype(np.int8)
df['is_q4'] = df['quarter'].isin([4]).astype(np.int8)

# 2) Ordinal encoding для естественно упорядоченных категорий
store_age_map = {
    'Новый': 0,
    'Средний по возрасту': 1,
    'Открыт давно': 2
}

area_map = {
    'Маленький': 0,
    'Средний': 1,
    'Большой': 2,
    'Очень большой': 3
}

df['store_age_ord'] = df['store_age_cat'].map(store_age_map).astype(np.int8)
df['area_ord'] = df['area_cat'].map(area_map).astype(np.int8)

# Для CatBoost оставим исходные как category
for col in ['city', 'region', 'store_age_cat', 'area_cat']:
    df[col] = df[col].astype('category')

# 3) Frequency encoding для высококардинальных категорий
for col in ['city', 'region']:
    freq = df[col].value_counts(dropna=False)
    df[f'{col}_freq'] = df[col].map(freq).astype(np.int32)

# 4) Лог-преобразования для сильно скошенных числовых статических признаков
log_cols = [
    'population',
    'households',
    'foot_traffic',
    'car_traffic',
    'marketplaces',
    'medical',
    'schools',
    'stops',
    'grocery_competitors',
    'pyaterochka_nearby',
    'n_kass'
]

for col in log_cols:
    df[f'log1p_{col}'] = np.log1p(df[col].clip(lower=0))

# 5) Динамические признаки: лаги по магазину
#    Используем только прошлые месяцы
dyn_cols = ['rto', 'avg_promo', 'avg_items', 'avg_cancels', 'work_hours']
lag_list = [1, 2, 3, 6, 12, 24]

for col in dyn_cols:
    grp = df.groupby('new_id', sort=False)[col]

    # Лаги
    for lag in lag_list:
        df[f'{col}_lag_{lag}'] = grp.shift(lag)

    # Rolling statistics по прошлым значениям
    shifted = grp.shift(1)
    for win in [3, 6, 12]:
        df[f'{col}_roll_mean_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).mean()
        )
        df[f'{col}_roll_std_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).std(ddof=0)
        )
        df[f'{col}_roll_median_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).median()
        )
        df[f'{col}_roll_min_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).min()
        )
        df[f'{col}_roll_max_{win}'] = shifted.groupby(df['new_id'], sort=False).transform(
            lambda x: x.rolling(win, min_periods=1).max()
        )


# 6) Сильные derived-features для RTO
df['rto_diff_1'] = df['rto_lag_1'] - df['rto_lag_2']
df['rto_diff_3'] = df['rto_lag_1'] - df['rto_lag_3']
df['rto_diff_12'] = df['rto_lag_1'] - df['rto_lag_12']
df['rto_diff_24'] = df['rto_lag_12'] - df['rto_lag_24']

df['rto_pct_change_1'] = df['rto_diff_1'] / (df['rto_lag_2'].abs() + eps)
df['rto_pct_change_12'] = df['rto_diff_12'] / (df['rto_lag_12'].abs() + eps)
df['rto_pct_change_24'] = df['rto_diff_24'] / (df['rto_lag_24'].abs() + eps)

df['rto_ratio_to_3m_mean'] = df['rto_lag_1'] / (df['rto_roll_mean_3'] + eps)
df['rto_ratio_to_6m_mean'] = df['rto_lag_1'] / (df['rto_roll_mean_6'] + eps)
df['rto_ratio_to_12m_mean'] = df['rto_lag_1'] / (df['rto_roll_mean_12'] + eps)

df['rto_ratio_mom'] = df['rto_lag_1'] / (df['rto_lag_2'] + eps)
df['rto_ratio_yoy'] = df['rto_lag_1'] / (df['rto_lag_12'] + eps)
df['rto_ratio_2y'] = df['rto_lag_12'] / (df['rto_lag_24'] + eps)

# Экспоненциальное сглаживание
df['rto_ewm_3'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).ewm(span=3, adjust=False).mean()
)
df['rto_ewm_6'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).ewm(span=6, adjust=False).mean()
)

# Расширяющееся среднее / std
df['rto_exp_mean'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)
df['rto_exp_std'] = df.groupby('new_id', sort=False)['rto'].transform(
    lambda x: x.shift(1).expanding(min_periods=2).std()
)


# 7) Несколько полезных derived-фич для динамических операционных метрик
for col in ['avg_promo', 'avg_items', 'avg_cancels', 'work_hours']:
    df[f'{col}_mom_ratio'] = df[f'{col}_lag_1'] / (df[f'{col}_lag_2'] + eps)
    df[f'{col}_yoy_ratio'] = df[f'{col}_lag_1'] / (df[f'{col}_lag_12'] + eps)
    df[f'{col}_to_3m_mean'] = df[f'{col}_lag_1'] / (df[f'{col}_roll_mean_3'] + eps)


new_feat_cols = [
    c for c in df.columns
    if (
        'lag_' in c or 'roll_' in c or 'ewm_' in c or 'exp_' in c or
        c.endswith('_ord') or c.endswith('_freq') or
        c.startswith('log1p_') or c in ['time_idx', 'quarter', 'month_sin', 'month_cos', 'is_december', 'is_q4']
    )
]

print("Новых признаков:", len(new_feat_cols))
print("Пример новых признаков:")
print(df[new_feat_cols].head(3).T.head(40))

In [ ]:
# 1) Производные статические признаки

# Отношения РТО к характеристикам локации (используем lag_1, чтобы не было leakage)
df['rto_per_capita'] = df['rto_lag_1'] / (df['population'] + 1)
df['rto_per_household'] = df['rto_lag_1'] / (df['households'] + 1)
df['rto_per_kassa'] = df['rto_lag_1'] / (df['n_kass'] + 1)

# Конкурентная среда
df['pyaterochka_share'] = df['pyaterochka_nearby'] / (df['grocery_competitors'] + 1)
df['competitors_per_capita'] = df['grocery_competitors'] / (df['population'] + 1)

# Трафик
df['traffic_total'] = df['foot_traffic'] + df['car_traffic']
df['traffic_per_kassa'] = df['traffic_total'] / (df['n_kass'] + 1)
df['foot_traffic_share'] = df['foot_traffic'] / (df['traffic_total'] + 1)

# Инфраструктура (интегральный показатель «проходимости» локации)
df['infra_score'] = (
    df['marketplaces'] + df['medical'] + df['schools'] + df['stops']
)
df['log1p_infra_score'] = np.log1p(df['infra_score'])
df['infra_per_capita'] = df['infra_score'] / (df['population'] + 1)

# Площадь x трафик (прокси «потенциала» точки)
df['area_x_traffic'] = df['area_ord'] * df['traffic_total']
df['area_x_kass'] = df['area_ord'] * df['n_kass']
df['kass_per_area'] = df['n_kass'] / (df['area_ord'] + 1)


# 2) Агрегаты по региону (на основе lag_1, без leakage)
for agg_col in ['region', 'city']:
    grp = df.groupby([agg_col, 'date'])['rto_lag_1']

    df[f'{agg_col}_rto_median'] = grp.transform('median')
    df[f'{agg_col}_rto_mean'] = grp.transform('mean')
    df[f'{agg_col}_rto_std'] = grp.transform('std')

    # Отклонение магазина от медианы по группе
    df[f'{agg_col}_rto_dev'] = df['rto_lag_1'] - df[f'{agg_col}_rto_median']

    # Ранг магазина внутри группы (перцентиль)
    df[f'{agg_col}_rto_rank'] = grp.rank(pct=True)


# 3) Агрегаты по (регион x месяц) — сезонность региона

region_month = df.groupby(['region', 'month'])['rto_lag_1']
df['region_month_median'] = region_month.transform('median')
df['region_month_dev'] = df['rto_lag_1'] - df['region_month_median']


# 4) Target Encoding: регион и город
#    Считаем на ПОЛНЫХ данных (потом при обучении будем делать OOF)
#    Сейчас — грубый вариант для feature engineering
#    Точный OOF target encoding сделаем в блоке обучения


# Пока используем простой вариант — среднее РТО по группе
# (для бустингов это уже полезно, а OOF-версию добавим позже)
for col in ['region', 'city', 'store_age_cat', 'area_cat']:
    te = df.groupby(col)['rto'].transform('mean')
    df[f'te_{col}_mean'] = te
    te_med = df.groupby(col)['rto'].transform('median')
    df[f'te_{col}_median'] = te_med


# 5) Проверка
new_cols_block2 = [
    'rto_per_capita', 'rto_per_household', 'rto_per_kassa',
    'pyaterochka_share', 'competitors_per_capita',
    'traffic_total', 'traffic_per_kassa', 'foot_traffic_share',
    'infra_score', 'log1p_infra_score', 'infra_per_capita',
    'area_x_traffic', 'area_x_kass', 'kass_per_area',
    'region_rto_median', 'region_rto_mean', 'region_rto_std',
    'region_rto_dev', 'region_rto_rank',
    'city_rto_median', 'city_rto_mean', 'city_rto_std',
    'city_rto_dev', 'city_rto_rank',
    'region_month_median', 'region_month_dev',
    'te_region_mean', 'te_region_median',
    'te_city_mean', 'te_city_median',
    'te_store_age_cat_mean', 'te_store_age_cat_median',
    'te_area_cat_mean', 'te_area_cat_median',
]

print(f"Новых признаков в блоке 2: {len(new_cols_block2)}")
print(f"Общее число колонок: {df.shape[1]}")
print(f"\nПримеры значений (магазин 0, последняя дата):")
print(df[df['new_id'] == 0][new_cols_block2].tail(1).T)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# 1) Подготовка таргета

# Сначала создадим колонку с РТО за тот же месяц прошлого года
df['rto_same_month_last_year'] = df.groupby('new_id')['rto'].shift(12)

# Вариант 1: year-over-year ratio (оптимален для MAPE)
df['target_ratio'] = df['rto'] / (df['rto_same_month_last_year'] + 1e-6)

# Вариант 2: month-over-month ratio
df['target_mom_ratio'] = df['rto'] / (df['rto_lag_1'] + 1e-6)

# Вариант 3: log transformation (классический)
df['target_log'] = np.log1p(df['rto'])

# Также сохраним абсолютный таргет для MAPE
df['target_absolute'] = df['rto']

# 2) Определяем период обучения

# Чтобы использовать lag_12, нужна история с января 2023
# Обучаем с января 2024 по февраль 2025 (14 месяцев)
train_mask = (df['date'] >= '2024-01-01') & (df['date'] <= '2025-02-01')
train_df = df[train_mask].copy()

print(f"Обучающих строк: {len(train_df)}")
print(f"Период: {train_df['date'].min()} - {train_df['date'].max()}")
print(f"Уникальных магазинов: {train_df['new_id'].nunique()}")

# 3) TimeSeriesSplit валидация

# Разделяем по времени, чтобы имитировать реальное прогнозирование
# 4 фолда: валидация на каждом последнем месяце
dates = sorted(train_df['date'].unique())
print(f"\nВсе даты в train: {[str(d.date()) for d in dates]}")

# Создадим 4 фолда, где валидация = последний месяц
tscv_folds = []
for i in range(1, 5):
    val_date = dates[-i]
    train_dates = dates[:-i]

    train_idx = train_df[train_df['date'].isin(train_dates)].index
    val_idx = train_df[train_df['date'] == val_date].index

    tscv_folds.append((train_idx, val_idx))

    print(f"Фолд {4-i+1}: train {len(train_idx)} строк, val {len(val_idx)} строк на {val_date.date()}")

# 4) Формирование тестового набора (март 2025)

# Создаем DataFrame для теста
test_df = df[df['new_id'].isin(df['new_id'].unique())].drop_duplicates('new_id')[['new_id']].copy()
test_df['date'] = pd.Timestamp('2025-03-01')
test_df['year'] = 2025
test_df['month'] = 3

print(f"\nТестовых строк: {len(test_df)}")

# 5) Подготовка фичей для теста

# Для каждого магазина в тесте нужно взять актуальные лаги из последних данных
# Получаем последнюю строку каждого магазина из исходных данных
last_records = df.sort_values('date').groupby('new_id').last().reset_index()

# Копируем нужные признаки в test_df
feature_cols_to_copy = [
    # Статические признаки
    'store_age_cat', 'area_cat', 'city', 'region',
    'population', 'households', 'foot_traffic', 'car_traffic',
    'marketplaces', 'medical', 'schools', 'stops',
    'grocery_competitors', 'pyaterochka_nearby', 'n_kass', 'alcohol_flag',

    # Кодированные версии
    'store_age_ord', 'area_ord', 'city_freq', 'region_freq',

    # Логарифмы
    'log1p_population', 'log1p_households', 'log1p_foot_traffic',
    'log1p_car_traffic', 'log1p_marketplaces', 'log1p_medical',
    'log1p_schools', 'log1p_stops', 'log1p_grocery_competitors',
    'log1p_pyaterochka_nearby', 'log1p_n_kass',

    # Календарные (для марта 2025)
    'quarter', 'month_sin', 'month_cos', 'is_december', 'is_q4'
]

# Копируем статические признаки
for col in feature_cols_to_copy:
    if col in last_records.columns:
        test_df[col] = test_df['new_id'].map(last_records.set_index('new_id')[col])

# Обновляем календарные признаки для марта 2025
test_df['quarter'] = 1  # март - Q1
test_df['month_sin'] = np.sin(2 * np.pi * 3 / 12)
test_df['month_cos'] = np.cos(2 * np.pi * 3 / 12)
test_df['is_december'] = 0
test_df['is_q4'] = 0

# 6) Функция для создания лагов в тесте

def create_test_lags(test_df, full_df):
    """Создает лаги для тестового набора (март 2025)"""

    test_df = test_df.copy()

    # Для каждого магазина в тесте берем значения из последних доступных месяцев
    for col in ['rto', 'avg_promo', 'avg_items', 'avg_cancels', 'work_hours']:
        # Получаем историю магазина
        store_history = full_df[full_df['new_id'].isin(test_df['new_id'])]

        # lag_1 = февраль 2025
        lag1_vals = store_history[store_history['date'] == '2025-02-01'][['new_id', col]]
        lag1_vals = lag1_vals.rename(columns={col: f'{col}_lag_1'})
        test_df = test_df.merge(lag1_vals, on='new_id', how='left')

        # lag_2 = январь 2025
        lag2_vals = store_history[store_history['date'] == '2025-01-01'][['new_id', col]]
        lag2_vals = lag2_vals.rename(columns={col: f'{col}_lag_2'})
        test_df = test_df.merge(lag2_vals, on='new_id', how='left')

        # lag_3 = декабрь 2024
        lag3_vals = store_history[store_history['date'] == '2024-12-01'][['new_id', col]]
        lag3_vals = lag3_vals.rename(columns={col: f'{col}_lag_3'})
        test_df = test_df.merge(lag3_vals, on='new_id', how='left')

        # lag_12 = март 2024
        lag12_vals = store_history[store_history['date'] == '2024-03-01'][['new_id', col]]
        lag12_vals = lag12_vals.rename(columns={col: f'{col}_lag_12'})
        test_df = test_df.merge(lag12_vals, on='new_id', how='left')

        # Rolling mean 3 (ноябрь 2024 - январь 2025)
        # Считаем вручную
        for win_name, months in [('3', ['2024-11-01', '2024-12-01', '2025-01-01']),
                                 ('6', ['2024-08-01', '2024-09-01', '2024-10-01',
                                        '2024-11-01', '2024-12-01', '2025-01-01'])]:

            win_vals = store_history[store_history['date'].isin(months)].groupby('new_id')[col].mean().reset_index()
            win_vals = win_vals.rename(columns={col: f'{col}_roll_mean_{win_name}'})
            test_df = test_df.merge(win_vals, on='new_id', how='left')

    return test_df

# Создаем лаги для теста
test_df = create_test_lags(test_df, df)

print(f"\nТестовый набор после создания лагов:")
print(f"Размер: {test_df.shape}")
print(f"Колонки (первые 20): {list(test_df.columns[:20])}")


# 7) Проверка данных

print("\n=== Проверка обучающих данных ===")
print(f"Пропуски в target_ratio: {train_df['target_ratio'].isna().sum()}")
print(f"Пропуски в rto_lag_1: {train_df['rto_lag_1'].isna().sum()}")
print(f"Пропуски в rto_lag_12: {train_df['rto_lag_12'].isna().sum()}")

print("\n=== Статистика таргетов ===")
for target in ['target_ratio', 'target_mom_ratio', 'target_log']:
    print(f"{target}: mean={train_df[target].mean():.4f}, std={train_df[target].std():.4f}")

print("\n=== Пример одного магазина (new_id=0) ===")
store_example = train_df[train_df['new_id'] == 0].sort_values('date')
print(store_example[['date', 'rto', 'rto_same_month_last_year', 'target_ratio']].tail())

print("\n=== Тестовый пример (первый магазин) ===")
print(test_df.iloc[0][['new_id', 'date', 'rto_lag_1', 'rto_lag_12',
                       'avg_promo_lag_1', 'avg_items_lag_1']].to_dict())

In [ ]:
eps = 1e-6

# 1) Пересобираем test_df для марта 2025 ПРАВИЛЬНО
#    Берем февральский срез как базу статических признаков
#    и достраиваем лаги/роллинги из полной истории


def build_march_test_from_history(df):
    feb_date = pd.Timestamp('2025-02-01')
    march_date = pd.Timestamp('2025-03-01')

    static_cols = [
        'new_id',
        'store_age_cat', 'area_cat', 'city', 'region',
        'population', 'households',
        'foot_traffic', 'car_traffic',
        'marketplaces', 'medical', 'schools', 'stops',
        'grocery_competitors', 'pyaterochka_nearby',
        'n_kass', 'alcohol_flag',
        'store_age_ord', 'area_ord',
        'city_freq', 'region_freq',
        'log1p_population', 'log1p_households',
        'log1p_foot_traffic', 'log1p_car_traffic',
        'log1p_marketplaces', 'log1p_medical',
        'log1p_schools', 'log1p_stops',
        'log1p_grocery_competitors', 'log1p_pyaterochka_nearby',
        'log1p_n_kass'
    ]

    test_df = (
        df[df['date'] == feb_date][static_cols]
        .drop_duplicates('new_id')
        .copy()
        .reset_index(drop=True)
    )

    # Календарь для марта 2025
    test_df['date'] = march_date
    test_df['year'] = 2025
    test_df['month'] = 3
    test_df['time_idx'] = (2025 - df['Год'].min()) * 12 + (3 - 1)
    test_df['quarter'] = 1
    test_df['month_sin'] = np.sin(2 * np.pi * 3 / 12)
    test_df['month_cos'] = np.cos(2 * np.pi * 3 / 12)
    test_df['is_december'] = 0
    test_df['is_q4'] = 0

    dyn_cols = ['rto', 'avg_promo', 'avg_items', 'avg_cancels', 'work_hours']

    lag_dates = {
        1: pd.Timestamp('2025-02-01'),
        2: pd.Timestamp('2025-01-01'),
        3: pd.Timestamp('2024-12-01'),
        6: pd.Timestamp('2024-09-01'),
        12: pd.Timestamp('2024-03-01'),
    }

    # Даты для rolling-окон для марта 2025:
    # roll_3  = Dec 2024, Jan 2025, Feb 2025
    # roll_6  = Sep 2024 ... Feb 2025
    # roll_12 = Mar 2024 ... Feb 2025
    rolling_dates = {
        3: pd.date_range(end=pd.Timestamp('2025-02-01'), periods=3, freq='MS'),
        6: pd.date_range(end=pd.Timestamp('2025-02-01'), periods=6, freq='MS'),
        12: pd.date_range(end=pd.Timestamp('2025-02-01'), periods=12, freq='MS'),
    }

    for col in dyn_cols:
        pivot = df.pivot(index='new_id', columns='date', values=col)
        pivot = pivot.reindex(index=test_df['new_id'])

        # Лаги
        for lag, lag_date in lag_dates.items():
            test_df[f'{col}_lag_{lag}'] = pivot[lag_date].values

        # Rolling statistics
        for win, dates in rolling_dates.items():
            mat = pivot.reindex(columns=dates)

            test_df[f'{col}_roll_mean_{win}'] = mat.mean(axis=1).values
            test_df[f'{col}_roll_std_{win}'] = mat.std(axis=1, ddof=0).values
            test_df[f'{col}_roll_median_{win}'] = mat.median(axis=1).values
            test_df[f'{col}_roll_min_{win}'] = mat.min(axis=1).values
            test_df[f'{col}_roll_max_{win}'] = mat.max(axis=1).values

    # 2) Derived-features для RTO

    test_df['rto_diff_1'] = test_df['rto_lag_1'] - test_df['rto_lag_2']
    test_df['rto_diff_3'] = test_df['rto_lag_1'] - test_df['rto_lag_3']
    test_df['rto_diff_12'] = test_df['rto_lag_1'] - test_df['rto_lag_12']

    test_df['rto_pct_change_1'] = test_df['rto_diff_1'] / (test_df['rto_lag_2'].abs() + eps)
    test_df['rto_pct_change_12'] = test_df['rto_diff_12'] / (test_df['rto_lag_12'].abs() + eps)

    test_df['rto_ratio_to_3m_mean'] = test_df['rto_lag_1'] / (test_df['rto_roll_mean_3'] + eps)
    test_df['rto_ratio_to_6m_mean'] = test_df['rto_lag_1'] / (test_df['rto_roll_mean_6'] + eps)
    test_df['rto_ratio_to_12m_mean'] = test_df['rto_lag_1'] / (test_df['rto_roll_mean_12'] + eps)

    test_df['rto_ratio_mom'] = test_df['rto_lag_1'] / (test_df['rto_lag_2'] + eps)
    test_df['rto_ratio_yoy'] = test_df['rto_lag_1'] / (test_df['rto_lag_12'] + eps)


    # 3) Derived-features для операционных метрик

    for col in ['avg_promo', 'avg_items', 'avg_cancels', 'work_hours']:
        test_df[f'{col}_mom_ratio'] = test_df[f'{col}_lag_1'] / (test_df[f'{col}_lag_2'] + eps)
        test_df[f'{col}_yoy_ratio'] = test_df[f'{col}_lag_1'] / (test_df[f'{col}_lag_12'] + eps)
        test_df[f'{col}_to_3m_mean'] = test_df[f'{col}_lag_1'] / (test_df[f'{col}_roll_mean_3'] + eps)


    # 4) Гео/социальные взаимодействия

    test_df['rto_per_capita'] = test_df['rto_lag_1'] / (test_df['population'] + 1)
    test_df['rto_per_household'] = test_df['rto_lag_1'] / (test_df['households'] + 1)
    test_df['rto_per_kassa'] = test_df['rto_lag_1'] / (test_df['n_kass'] + 1)

    test_df['pyaterochka_share'] = test_df['pyaterochka_nearby'] / (test_df['grocery_competitors'] + 1)
    test_df['competitors_per_capita'] = test_df['grocery_competitors'] / (test_df['population'] + 1)

    test_df['traffic_total'] = test_df['foot_traffic'] + test_df['car_traffic']
    test_df['traffic_per_kassa'] = test_df['traffic_total'] / (test_df['n_kass'] + 1)
    test_df['foot_traffic_share'] = test_df['foot_traffic'] / (test_df['traffic_total'] + 1)

    test_df['infra_score'] = (
        test_df['marketplaces'] + test_df['medical'] + test_df['schools'] + test_df['stops']
    )
    test_df['log1p_infra_score'] = np.log1p(test_df['infra_score'])
    test_df['infra_per_capita'] = test_df['infra_score'] / (test_df['population'] + 1)

    test_df['area_x_traffic'] = test_df['area_ord'] * test_df['traffic_total']
    test_df['area_x_kass'] = test_df['area_ord'] * test_df['n_kass']
    test_df['kass_per_area'] = test_df['n_kass'] / (test_df['area_ord'] + 1)


    # 5) Групповые агрегаты для марта 2025
    #    Они должны строиться на rto_lag_1 (= февраль 2025)

    for agg_col in ['region', 'city']:
        grp = test_df.groupby(agg_col)['rto_lag_1']

        test_df[f'{agg_col}_rto_median'] = grp.transform('median')
        test_df[f'{agg_col}_rto_mean'] = grp.transform('mean')
        test_df[f'{agg_col}_rto_std'] = grp.transform('std')
        test_df[f'{agg_col}_rto_dev'] = test_df['rto_lag_1'] - test_df[f'{agg_col}_rto_median']
        test_df[f'{agg_col}_rto_rank'] = grp.rank(pct=True)

    test_df['region_month_median'] = test_df.groupby(['region', 'month'])['rto_lag_1'].transform('median')
    test_df['region_month_dev'] = test_df['rto_lag_1'] - test_df['region_month_median']

    # Инфинити -> NaN
    test_df = test_df.replace([np.inf, -np.inf], np.nan)

    return test_df


# Пересобираем test
test_df = build_march_test_from_history(df)

print("test_df shape:", test_df.shape)
print("Пример колонок:", test_df.columns[:30].tolist())
print(test_df[['new_id', 'date', 'rto_lag_1', 'rto_lag_2', 'rto_lag_3', 'rto_lag_6', 'rto_lag_12']].head())

In [ ]:
train_df = train_df.sort_values(['date', 'new_id']).reset_index(drop=True)

# На всякий случай убираем inf
train_df = train_df.replace([np.inf, -np.inf], np.nan)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

In [ ]:
# te_* пока НЕ используем это leakage
te_cols = [c for c in train_df.columns if c.startswith('te_')]

drop_cols = {
    'rto',
    'target_ratio',
    'target_mom_ratio',
    'target_log',
    'target_absolute',
    'rto_same_month_last_year',
    'date',
    'Год',
    'Месяц',

    # текущие динамические признаки марта нам неизвестны
    'avg_promo',
    'avg_items',
    'avg_cancels',
    'work_hours',
}

drop_cols = drop_cols.union(set(te_cols))

# Оставляем только общие признаки между train и test
common_cols = sorted((set(train_df.columns) & set(test_df.columns)) - drop_cols)

# Отдельно:
# numeric features -> для LightGBM / XGBoost / Ridge / ExtraTrees
# cat features     -> для CatBoost
cat_features = ['city', 'region', 'store_age_cat', 'area_cat']
cat_features = [c for c in cat_features if c in common_cols]

numeric_features = [
    c for c in common_cols
    if c not in cat_features and pd.api.types.is_numeric_dtype(train_df[c])
]

catboost_features = numeric_features + cat_features

print("Количество numeric_features:", len(numeric_features))
print("Количество catboost_features:", len(catboost_features))
print("Категориальные для CatBoost:", cat_features[:])

In [ ]:
X_train_num = train_df[numeric_features].copy()
X_test_num = test_df[numeric_features].copy()

X_train_cat = train_df[catboost_features].copy()
X_test_cat = test_df[catboost_features].copy()

y_ratio = train_df['target_ratio'].copy()
y_abs = train_df['target_absolute'].copy()
y_log = train_df['target_log'].copy()

print("X_train_num:", X_train_num.shape)
print("X_test_num :", X_test_num.shape)
print("X_train_cat:", X_train_cat.shape)
print("X_test_cat :", X_test_cat.shape)

In [ ]:
val_dates = [
    pd.Timestamp('2024-11-01'),
    pd.Timestamp('2024-12-01'),
    pd.Timestamp('2025-01-01'),
    pd.Timestamp('2025-02-01'),
]

folds = []
for i, val_date in enumerate(val_dates, 1):
    tr_idx = np.where(train_df['date'] < val_date)[0]
    va_idx = np.where(train_df['date'] == val_date)[0]
    folds.append((tr_idx, va_idx))
    print(f'Fold {i}: train={len(tr_idx)}, valid={len(va_idx)}, val_date={val_date.date()}')

In [ ]:
print("\n=== Топ пропусков в train numeric ===")
print(
    X_train_num.isna().mean().sort_values(ascending=False).head(20)
)

print("\n=== Топ пропусков в test numeric ===")
print(
    X_test_num.isna().mean().sort_values(ascending=False).head(20)
)

print("\n=== Проверка таргета ===")
print("target_ratio stats:")
print(y_ratio.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("\n=== Проверка тестовых лагов ===")
print(
    test_df[['rto_lag_1', 'rto_lag_2', 'rto_lag_3', 'rto_lag_6', 'rto_lag_12']].describe()
)

In [ ]:
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

EPS = 1e-6

# Метрики

def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_pred - y_true) / np.clip(np.abs(y_true), EPS, None)) * 100

def competition_points(mape_value):
    return 100 * ((100 - min(mape_value, 100)) / 100) ** 2

# Добавим новые таргеты
train_df['target_log_ratio'] = np.log(train_df['target_ratio'].clip(lower=EPS))
train_df['target_log_abs2'] = np.log(train_df['target_absolute'].clip(lower=EPS))

print("target_log_ratio stats:")
print(train_df['target_log_ratio'].describe())

print("\ntarget_log_abs2 stats:")
print(train_df['target_log_abs2'].describe())

In [ ]:
cluster_features = [
    'log1p_population',
    'log1p_households',
    'log1p_foot_traffic',
    'log1p_car_traffic',
    'log1p_n_kass',
    'area_ord',
    'store_age_ord',
    'log1p_grocery_competitors',
    'log1p_marketplaces',
    'log1p_medical',
    'log1p_schools',
    'log1p_stops',
    'log1p_pyaterochka_nearby',
]

store_static = df.drop_duplicates('new_id')[['new_id'] + cluster_features].reset_index(drop=True)
print(f"Магазинов для кластеризации: {len(store_static)}")

scaler = StandardScaler()
X_cluster = scaler.fit_transform(store_static[cluster_features])

cluster_ks = [5, 10, 20, 50]

for k in cluster_ks:
    print(f"\nKMeans (k={k})...")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    store_static[f'cluster_{k}'] = kmeans.fit_predict(X_cluster)
    counts = pd.Series(store_static[f'cluster_{k}']).value_counts().sort_index()
    print(f"  min={counts.min()}, max={counts.max()}, median={counts.median():.0f}")

cluster_cols = [f'cluster_{k}' for k in cluster_ks]
cluster_map = store_static.set_index('new_id')[cluster_cols]

for col in cluster_cols:
    train_df[col] = train_df['new_id'].map(cluster_map[col]).astype('int32')
    test_df[col] = test_df['new_id'].map(cluster_map[col]).astype('int32')

print(f"\nКластерные колонки добавлены")

In [ ]:
def add_cluster_te_features(train_df, test_df, cluster_col):
    # Медиана и среднее РТО кластера (на основе lag_1)
    train_df[f'{cluster_col}_rto_lag1_median'] = (
        train_df.groupby([cluster_col, 'date'])['rto_lag_1'].transform('median')
    )
    test_df[f'{cluster_col}_rto_lag1_median'] = (
        test_df.groupby(cluster_col)['rto_lag_1'].transform('median')
    )

    train_df[f'{cluster_col}_rto_lag1_mean'] = (
        train_df.groupby([cluster_col, 'date'])['rto_lag_1'].transform('mean')
    )
    test_df[f'{cluster_col}_rto_lag1_mean'] = (
        test_df.groupby(cluster_col)['rto_lag_1'].transform('mean')
    )

    # Год назад
    train_df[f'{cluster_col}_rto_lag12_median'] = (
        train_df.groupby([cluster_col, 'date'])['rto_lag_12'].transform('median')
    )
    test_df[f'{cluster_col}_rto_lag12_median'] = (
        test_df.groupby(cluster_col)['rto_lag_12'].transform('median')
    )

    # YoY рост по кластеру
    train_df[f'{cluster_col}_yoy_growth_median'] = (
        train_df.groupby([cluster_col, 'date'])['rto_ratio_yoy'].transform('median')
    )
    test_df[f'{cluster_col}_yoy_growth_median'] = (
        test_df.groupby(cluster_col)['rto_ratio_yoy'].transform('median')
    )

    # Отклонение от кластера
    train_df[f'{cluster_col}_rto_deviation'] = (
        train_df['rto_lag_1'] - train_df[f'{cluster_col}_rto_lag1_median']
    )
    test_df[f'{cluster_col}_rto_deviation'] = (
        test_df['rto_lag_1'] - test_df[f'{cluster_col}_rto_lag1_median']
    )

    # Ранг
    train_df[f'{cluster_col}_rank_in_cluster'] = (
        train_df.groupby([cluster_col, 'date'])['rto_lag_1'].rank(pct=True)
    )
    test_df[f'{cluster_col}_rank_in_cluster'] = (
        test_df.groupby(cluster_col)['rto_lag_1'].rank(pct=True)
    )

    # Сезонность кластера
    train_df[f'{cluster_col}_month_seasonal'] = (
        train_df.groupby([cluster_col, 'month'])['rto_lag_12'].transform('median')
    )
    test_df[f'{cluster_col}_month_seasonal'] = (
        test_df.groupby(cluster_col)['rto_lag_12'].transform('median')
    )

for k in cluster_ks:
    print(f"Создание TE-фичей для cluster_{k}...")
    add_cluster_te_features(train_df, test_df, f'cluster_{k}')

train_df = train_df.replace([np.inf, -np.inf], np.nan)
test_df = test_df.replace([np.inf, -np.inf], np.nan)

cluster_te_cols = [
    c for c in train_df.columns
    if c.startswith('cluster_') and c not in cluster_cols
]
print(f"\nДобавлено TE-фичей: {len(cluster_te_cols)}")

In [ ]:
all_cluster_features = cluster_cols + cluster_te_cols

numeric_features_v2 = numeric_features + all_cluster_features
numeric_features_v2 = [
    c for c in numeric_features_v2
    if c in train_df.columns and c in test_df.columns
]

print(f"Признаков было: {len(numeric_features)}")
print(f"Признаков стало: {len(numeric_features_v2)}")

X_train_num_v2 = train_df[numeric_features_v2].copy()
X_test_num_v2 = test_df[numeric_features_v2].copy()

print(f"X_train_num_v2: {X_train_num_v2.shape}")
print(f"X_test_num_v2:  {X_test_num_v2.shape}")

In [ ]:
best_params_logratio = {
    'objective': 'regression',
    'metric': 'l1',
    'n_estimators': 10000,
    'learning_rate': 0.005160968768096091,
    'num_leaves': 54,
    'max_depth': 8,
    'min_child_samples': 49,
    'min_child_weight': 0.009535400814595932,
    'subsample': 0.9233556350190333,
    'subsample_freq': 4,
    'colsample_bytree': 0.7926568065394248,
    'reg_alpha': 0.31423869800216053,
    'reg_lambda': 4.7942976434286,
    'min_split_gain': 0.0015407987938578227,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

In [ ]:
def run_lgb_oof(
    X_train, X_test, train_df, test_df, folds,
    target_col, mode='log_ratio', params=None, model_name='lgb'
):
    y = train_df[target_col].values

    oof_abs = np.full(len(X_train), np.nan)
    test_abs_folds = np.zeros((len(X_test), len(folds)))
    fold_scores = []
    feature_importance = []

    for fold_num, (tr_idx, va_idx) in enumerate(folds, 1):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric='l1',
            callbacks=[
                lgb.early_stopping(stopping_rounds=300, verbose=False),
                lgb.log_evaluation(period=0)
            ]
        )

        pred_va = model.predict(X_va, num_iteration=model.best_iteration_)
        pred_test = model.predict(X_test, num_iteration=model.best_iteration_)

        if mode == 'log_ratio':
            pred_va_abs = np.exp(pred_va) * train_df.iloc[va_idx]['rto_lag_12'].values
            pred_test_abs = np.exp(pred_test) * test_df['rto_lag_12'].values
        elif mode == 'log_abs':
            pred_va_abs = np.exp(pred_va)
            pred_test_abs = np.exp(pred_test)
        elif mode == 'mom_ratio':
            pred_va_abs = pred_va * train_df.iloc[va_idx]['rto_lag_1'].values
            pred_test_abs = pred_test * test_df['rto_lag_1'].values

        pred_va_abs = np.clip(pred_va_abs, 1e5, None)
        pred_test_abs = np.clip(pred_test_abs, 1e5, None)

        true_va_abs = train_df.iloc[va_idx]['target_absolute'].values
        fold_mape = mape(true_va_abs, pred_va_abs)

        print(f"{model_name} | Fold {fold_num} | best_iter={model.best_iteration_} | MAPE={fold_mape:.4f}")

        fold_scores.append(fold_mape)
        oof_abs[va_idx] = pred_va_abs
        test_abs_folds[:, fold_num - 1] = pred_test_abs

        fi = pd.DataFrame({
            'feature': X_train.columns,
            'importance': model.feature_importances_,
            'fold': fold_num
        })
        feature_importance.append(fi)

    covered_mask = ~np.isnan(oof_abs)
    oof_mape = mape(
        train_df.loc[covered_mask, 'target_absolute'].values,
        oof_abs[covered_mask]
    )

    print(f"\n{model_name} FINAL OOF MAPE: {oof_mape:.5f}")
    print(f"{model_name} Points: {competition_points(oof_mape):.5f}")

    fi_df = pd.concat(feature_importance, axis=0)
    fi_df = (
        fi_df.groupby('feature', as_index=False)['importance']
        .mean()
        .sort_values('importance', ascending=False)
    )

    return {
        'oof_abs': oof_abs,
        'covered_mask': covered_mask,
        'test_abs': test_abs_folds.mean(axis=1),
        'fold_scores': fold_scores,
        'oof_mape': oof_mape,
        'fi_df': fi_df,
    }

In [ ]:
print("LGB_log_ratio + CLUSTERS")

lgb_logratio_v3 = run_lgb_oof(
    X_train=X_train_num_v2,
    X_test=X_test_num_v2,
    train_df=train_df,
    test_df=test_df,
    folds=folds,
    target_col='target_log_ratio',
    mode='log_ratio',
    params=best_params_logratio,
    model_name='LGB_log_ratio_CLUSTER'
)

In [ ]:
sub = pd.DataFrame({'new_id': test_df['new_id'], 'rto': lgb_logratio_v3['test_abs']})
sub.to_csv('test.csv', index=False)